# 01 - Initial Data Exploration

This notebook loads the raw Online Retail dataset, answers the required data understanding questions, and writes `data_quality_summary.json` with the standardized schema.

In [ ]:
# Cell 1: Import libraries and set display options
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Cell 2: Load raw dataset
df = pd.read_csv('../data/raw/online_retail.csv', encoding='latin1')
print(f'Dataset Shape: {df.shape}')
df.head(10)

In [ ]:
# Cells 3-10: Q1-Q13 analysis

# Q1-Q4: basic statistics and uniques
date_min, date_max = df['InvoiceDate'].min(), df['InvoiceDate'].max()
print('Q1 - Date range:', date_min, 'to', date_max)
print('Q2 - Unique customers:', df['Customer ID'].nunique())
print('Q3 - Unique products:', df['StockCode'].nunique())
print('Q4 - Unique countries:', df['Country'].nunique())

# Q5-Q7: data quality
print('Q5 - Missing values per column:\n', df.isnull().sum())
missing_cust = df['Customer ID'].isnull().sum()
print('Q6 - % missing Customer ID:', missing_cust / len(df) * 100)
print('Q7 - Duplicate rows:', df.duplicated().sum())

# Q8-Q10: transaction patterns
print('Q8 - Quantity describe:\n', df['Quantity'].describe())
print('Q9 - Negative quantities:', (df['Quantity'] < 0).sum())
print('Q10 - Price describe:\n', df['Price'].describe())

# Q11-Q13: anomalies
print('Q11 - Cancelled invoices (start with C):', df['Invoice'].astype(str).str.startswith('C').sum())
print('Q12 - Zero or negative prices:', (df['Price'] <= 0).sum())
print('Q13 - Extremely high quantities (>1000):', (df['Quantity'] > 1000).sum())

In [ ]:
# Cell 11: Build standardized data_quality_summary.json
from pathlib import Path

summary = {
    'total_rows': int(len(df)),
    'total_columns': int(len(df.columns)),
    'missing_values': df.isnull().sum().to_dict(),
    'duplicate_rows': int(df.duplicated().sum()),
    'date_range': {
        'start': str(pd.to_datetime(df['InvoiceDate']).min().date()),
        'end': str(pd.to_datetime(df['InvoiceDate']).max().date()),
    },
    'negative_quantities': int((df['Quantity'] < 0).sum()),
    'cancelled_invoices': int(df['Invoice'].astype(str).str.startswith('C').sum()),
    'missing_customer_ids': int(df['Customer ID'].isnull().sum()),
}
summary['missing_customer_ids_percentage'] = round(summary['missing_customer_ids'] / summary['total_rows'] * 100, 1)

output_path = Path('../data/raw/data_quality_summary.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w') as f:
    json.dump(summary, f, indent=4, default=str)

print('data_quality_summary.json written to', output_path)